# Advanced Assignment Expressions (`:=`)
## A Tutorial-Style Problem Set with Complete Solutions

Assignment expressions were introduced in Python 3.8. They allow an expression to
**produce a value** and **bind that value to a name** at the same time.

This notebook develops the topic through a sequence of advanced problems. Each
problem is broken into small logical steps:

1. understand the situation,
2. inspect a straightforward implementation,
3. identify repetition or a correctness risk,
4. introduce an assignment expression only where it helps,
5. test the result,
6. discuss readability and best practices.

The goal is not to use the walrus operator everywhere. The goal is to recognize
the relatively small number of situations where it makes code clearer, safer, or
more efficient.

## Learning Objectives

By the end of this notebook, you should be able to:

- use `:=` safely in `if` and `while` conditions,
- avoid repeating expensive or state-changing expressions,
- understand evaluation order in comprehensions,
- distinguish missing values from valid falsy values,
- capture a useful value from a short-circuiting operation,
- reason about scope and lifetime of assignment-expression targets,
- identify precedence mistakes,
- decide when an ordinary assignment is more readable,
- test walrus-based refactorings for semantic equivalence.

## Notebook Conventions

Every example is deterministic and uses only the Python standard library.

Assertions are included throughout the notebook. An assertion is not merely a
demonstration: it is an executable statement of the behavior we expect.

Several examples deliberately show an unattractive or incorrect approach first.
Those examples are placed inside functions, strings, or `try`/`except` blocks so
that the notebook can still run from top to bottom.

In [1]:
import io
import json
import re
from collections import Counter
from typing import Any, Dict, Iterable, Iterator, List, Optional, Tuple

print("Notebook setup complete.")

Notebook setup complete.


# Part 1 — Reading and Reusing Values in Loops

## Problem 1 — Process Commands Until a Sentinel Appears

Suppose a command source produces raw strings. A command may contain surrounding
spaces, and a blank command should be ignored.

Processing must stop when the normalized command is `"quit"`, regardless of
capitalization.

The source can also become exhausted. Exhaustion is represented by `None`.

We therefore need to distinguish three cases:

- `None`: the source is exhausted,
- `""` after stripping: ignore the blank command,
- `"quit"` after stripping: stop intentionally.

### Step 1 — Build a deterministic command source

In a real program, the values might come from `input()`, a socket, or a message
queue. Here we use an iterator so that the example is repeatable.

In [2]:
raw_commands = iter([
    "  STATUS  ",
    "   ",
    "deploy api",
    "  ReStart Worker ",
    "QUIT",
    "this should never be processed",
])

### Step 2 — Write the ordinary version

This version is perfectly valid. Notice, however, that the loop contains a
separate assignment whose only purpose is to feed the condition and the body.

In [3]:
def process_commands_standard(source: Iterator[str]) -> List[str]:
    processed = []

    while True:
        raw = next(source, None)
        if raw is None:
            break

        command = raw.strip()
        if not command:
            continue

        if command.casefold() == "quit":
            break

        processed.append(command)

    return processed


standard_commands = process_commands_standard(iter([
    "  STATUS  ",
    "   ",
    "deploy api",
    "  ReStart Worker ",
    "QUIT",
    "ignored",
]))

standard_commands

['STATUS', 'deploy api', 'ReStart Worker']

### Step 3 — Move the read operation into the loop condition

The expression `next(source, None)` both produces the next item and tells us
whether the source is exhausted.

The parenthesized expression

```python
(raw := next(source, None))
```

binds the produced value to `raw`. Comparing it with `None` avoids confusing an
empty string with exhaustion.

In [4]:
def process_commands_walrus(source: Iterator[str]) -> List[str]:
    processed = []

    while (raw := next(source, None)) is not None:
        if not (command := raw.strip()):
            continue

        if command.casefold() == "quit":
            break

        processed.append(command)

    return processed


walrus_commands = process_commands_walrus(iter([
    "  STATUS  ",
    "   ",
    "deploy api",
    "  ReStart Worker ",
    "QUIT",
    "ignored",
]))

walrus_commands

['STATUS', 'deploy api', 'ReStart Worker']

### Step 4 — Verify semantic equivalence

A walrus-based refactoring should preserve behavior. We test both the expected
output and equivalence with the ordinary implementation.

In [5]:
expected_commands = ["STATUS", "deploy api", "ReStart Worker"]

assert standard_commands == expected_commands
assert walrus_commands == expected_commands
assert standard_commands == walrus_commands

print("Problem 1 checks passed.")

Problem 1 checks passed.


### Discussion

Two separate assignment expressions are used here:

- `raw` captures the value that controls loop termination.
- `command` captures the stripped text that is both tested and later reused.

This is a good use of `:=` because each assigned value is immediately relevant to
the surrounding condition.

The explicit comparison `is not None` is important. Writing

```python
while raw := next(source, None):
```

would stop on `""`, even though an empty command and source exhaustion are
different events.

## Problem 2 — Read Fixed-Size Binary Chunks

A binary stream returns `b""` when it reaches the end. We want to read chunks,
record their sizes, and compute a simple checksum.

This is a classic case where the value returned by a function both controls the
loop and is needed inside the loop body.

### Step 1 — Inspect the ordinary loop

In [6]:
def inspect_stream_standard(data: bytes, chunk_size: int) -> Tuple[List[int], int]:
    stream = io.BytesIO(data)
    sizes = []
    checksum = 0

    while True:
        chunk = stream.read(chunk_size)
        if chunk == b"":
            break

        sizes.append(len(chunk))
        checksum += sum(chunk)

    return sizes, checksum


inspect_stream_standard(b"assignment-expressions", 5)

([5, 5, 5, 5, 2], 2345)

### Step 2 — Use the stream's documented sentinel

For binary reads, `b""` specifically means end-of-file. A non-empty byte string
is truthy, so a compact `while` condition is appropriate.

In [7]:
def inspect_stream_walrus(data: bytes, chunk_size: int) -> Tuple[List[int], int]:
    stream = io.BytesIO(data)
    sizes = []
    checksum = 0

    while chunk := stream.read(chunk_size):
        sizes.append(len(chunk))
        checksum += sum(chunk)

    return sizes, checksum


stream_result = inspect_stream_walrus(b"assignment-expressions", 5)
stream_result

([5, 5, 5, 5, 2], 2345)

### Step 3 — Test boundaries

A robust solution should handle:

- empty input,
- input shorter than one chunk,
- input exactly divisible by the chunk size,
- a final partial chunk.

In [8]:
assert inspect_stream_walrus(b"", 4) == ([], 0)
assert inspect_stream_walrus(b"abc", 4) == ([3], sum(b"abc"))
assert inspect_stream_walrus(b"abcdefgh", 4)[0] == [4, 4]
assert inspect_stream_walrus(b"abcdefghi", 4)[0] == [4, 4, 1]

for payload in [b"", b"x", b"abcdef", b"assignment-expressions"]:
    assert inspect_stream_standard(payload, 5) == inspect_stream_walrus(payload, 5)

print("Problem 2 checks passed.")

Problem 2 checks passed.


### Alternative Tool — `iter(callable, sentinel)`

Python also supports the two-argument form of `iter`:

```python
for chunk in iter(lambda: stream.read(chunk_size), b""):
    ...
```

That form is excellent when the loop condition is exactly “call until a sentinel
is returned.” The walrus form is often preferable when the loop has additional
state or a more complex condition.

Best practice is not “always use `:=`.” Best practice is to choose the clearest
tool for the loop.

# Part 2 — Avoiding Repeated Work

## Problem 3 — Normalize Text Exactly Once in a Comprehension

Suppose normalization is moderately expensive. It trims a string, collapses
internal whitespace, converts it to lowercase, and rejects strings that contain
no alphanumeric characters.

We want normalized strings whose length is at least four characters.

A careless comprehension may normalize each accepted value twice.

### Step 1 — Create an instrumented normalizer

The counter lets us test how many times the function is evaluated.

In [9]:
normalization_calls = 0

def normalize_text(value: str) -> Optional[str]:
    global normalization_calls
    normalization_calls += 1

    cleaned = " ".join(value.split()).casefold()
    if not any(character.isalnum() for character in cleaned):
        return None
    return cleaned

### Step 2 — Observe the repeated-call version

The function appears in both the element expression and the filter. Accepted
values are evaluated twice.

In [10]:
raw_labels = [
    "  Alpha  Team ",
    " -- ",
    "API",
    "Data    Platform",
    " ML ",
    "Customer SUCCESS",
]

normalization_calls = 0

repeated_version = [
    normalize_text(raw)
    for raw in raw_labels
    if normalize_text(raw) is not None
    and len(normalize_text(raw)) >= 4
]

repeated_call_count = normalization_calls
repeated_version, repeated_call_count

(['alpha team', 'data platform', 'customer success'], 14)

The version above is even worse than a typical duplicated expression: accepted
values may be normalized three times.

Besides performance, repeated evaluation may be incorrect when a function reads
changing external state or has side effects.

### Step 3 — First improve the code with an ordinary loop

This is the readability baseline. The function is called once per input.

In [11]:
normalization_calls = 0
loop_version = []

for raw in raw_labels:
    normalized = normalize_text(raw)
    if normalized is not None and len(normalized) >= 4:
        loop_version.append(normalized)

loop_call_count = normalization_calls
loop_version, loop_call_count

(['alpha team', 'data platform', 'customer success'], 6)

### Step 4 — Express the same logic in one comprehension

The assignment must occur in the `if` clause because comprehension evaluation
happens in this order:

1. obtain the next loop item,
2. evaluate the filter,
3. evaluate the element expression.

The element expression can reuse `normalized` only after the filter has assigned
it.

In [12]:
normalization_calls = 0

walrus_version = [
    normalized
    for raw in raw_labels
    if (normalized := normalize_text(raw)) is not None
    and len(normalized) >= 4
]

walrus_call_count = normalization_calls
walrus_version, walrus_call_count

(['alpha team', 'data platform', 'customer success'], 6)

### Step 5 — Verify both correctness and evaluation count

In [13]:
expected_labels = [
    "alpha team",
    "data platform",
    "customer success",
]

assert loop_version == expected_labels
assert walrus_version == expected_labels
assert loop_version == walrus_version

assert loop_call_count == len(raw_labels)
assert walrus_call_count == len(raw_labels)
assert repeated_call_count > walrus_call_count

print({
    "inputs": len(raw_labels),
    "repeated_calls": repeated_call_count,
    "loop_calls": loop_call_count,
    "walrus_calls": walrus_call_count,
})

{'inputs': 6, 'repeated_calls': 14, 'loop_calls': 6, 'walrus_calls': 6}


### Discussion

This is one of the strongest use cases for assignment expressions:

```python
[
    normalized
    for raw in values
    if (normalized := normalize(raw)) is not None
]
```

The assigned name communicates meaning and prevents repeated computation.

Notice that we compare with `None` rather than relying on truthiness. That choice
documents the contract of `normalize_text`: `None` means rejection.

## Problem 4 — Parse Log Records with One Regular-Expression Match

Each valid line has this format:

```text
LEVEL|NNN|message
```

where `LEVEL` is `INFO`, `WARN`, or `ERROR`, and `NNN` is a three-digit code.

We want only valid non-`INFO` records. The match object is needed both to decide
whether a line is valid and to extract fields.

### Step 1 — Define the input and pattern

In [14]:
log_lines = [
    "INFO|200|startup complete",
    "WARN|301|cache nearing capacity",
    "malformed line",
    "ERROR|503|upstream unavailable",
    "DEBUG|100|unsupported level",
    "WARN|099|clock skew detected",
]

log_pattern = re.compile(
    r"(?P<level>INFO|WARN|ERROR)\|"
    r"(?P<code>\d{3})\|"
    r"(?P<message>.+)"
)

### Step 2 — Write a small extraction helper

Keeping extraction in a helper prevents a comprehension from becoming too dense.

In [15]:
def record_from_match(match: re.Match) -> Dict[str, Any]:
    return {
        "level": match.group("level"),
        "code": int(match.group("code")),
        "message": match.group("message"),
    }

### Step 3 — Capture the match object in the filter

The first condition assigns `match`. Because `and` short-circuits, the second
condition runs only when a match exists.

In [16]:
important_records = [
    record_from_match(match)
    for line in log_lines
    if (match := log_pattern.fullmatch(line)) is not None
    and match.group("level") != "INFO"
]

important_records

[{'level': 'WARN', 'code': 301, 'message': 'cache nearing capacity'},
 {'level': 'ERROR', 'code': 503, 'message': 'upstream unavailable'},
 {'level': 'WARN', 'code': 99, 'message': 'clock skew detected'}]

### Step 4 — Check the result

In [17]:
assert important_records == [
    {
        "level": "WARN",
        "code": 301,
        "message": "cache nearing capacity",
    },
    {
        "level": "ERROR",
        "code": 503,
        "message": "upstream unavailable",
    },
    {
        "level": "WARN",
        "code": 99,
        "message": "clock skew detected",
    },
]

print("Problem 4 checks passed.")

Problem 4 checks passed.


### Discussion

Without the assignment expression, a one-line comprehension would either call
`fullmatch` repeatedly or require a more complicated nested iteration.

The helper function is important. Assignment expressions can remove repetition,
but they should not be used as an excuse to pack parsing, conversion, validation,
and construction into one unreadable line.

# Part 3 — Sentinels, Missing Values, and Falsy Values

## Problem 5 — Validate Configuration Without Rejecting Zero

A configuration dictionary may contain a timeout in seconds.

Rules:

- a missing timeout is an error,
- `0` is valid and means “do not wait,”
- positive integers are valid,
- negative integers and non-integers are invalid.

A truthiness check is dangerous because `0` is falsy.

### Step 1 — See the tempting but incorrect condition

In [18]:
def validate_timeout_incorrect(config: Dict[str, Any]) -> str:
    if timeout := config.get("timeout"):
        return f"accepted: {timeout}"
    return "missing or invalid"


incorrect_examples = {
    "zero": validate_timeout_incorrect({"timeout": 0}),
    "five": validate_timeout_incorrect({"timeout": 5}),
    "missing": validate_timeout_incorrect({}),
}

incorrect_examples

{'zero': 'missing or invalid',
 'five': 'accepted: 5',
 'missing': 'missing or invalid'}

The configuration `{"timeout": 0}` is treated like a missing value. The problem
is not the assignment expression itself; the problem is using truthiness when
the domain gives `0` a legitimate meaning.

### Step 2 — Introduce a unique sentinel

A unique object lets us distinguish “key not present” from every possible stored
value.

In [19]:
MISSING = object()

def validate_timeout(config: Dict[str, Any]) -> int:
    if (timeout := config.get("timeout", MISSING)) is MISSING:
        raise ValueError("timeout is required")

    if isinstance(timeout, bool) or not isinstance(timeout, int):
        raise TypeError("timeout must be an integer")

    if timeout < 0:
        raise ValueError("timeout must be non-negative")

    return timeout

### Step 3 — Test valid and invalid cases

`bool` requires special attention because `bool` is a subclass of `int` in
Python. A strict integer validator often needs to reject booleans explicitly.

In [20]:
assert validate_timeout({"timeout": 0}) == 0
assert validate_timeout({"timeout": 5}) == 5

for bad_config, expected_exception in [
    ({}, ValueError),
    ({"timeout": -1}, ValueError),
    ({"timeout": 2.5}, TypeError),
    ({"timeout": True}, TypeError),
]:
    try:
        validate_timeout(bad_config)
    except expected_exception:
        pass
    else:
        raise AssertionError(f"Expected {expected_exception.__name__}: {bad_config}")

print("Problem 5 checks passed.")

Problem 5 checks passed.


### Best-Practice Rule

Use truthiness when the domain genuinely means “empty or false should stop.”

Use an explicit comparison when values such as `0`, `False`, `""`, `[]`, or
`b""` may be valid data.

Assignment expressions do not change this rule. They make it even more important
because assignment and testing appear in the same expression.

## Problem 6 — Consume Paginated Results Correctly

A page-fetching function returns either:

- `(items, next_token)`, or
- `None` when no page remains.

An empty page is valid. It may still contain a token for a later page.

Therefore, testing the page value by truthiness would be an unreliable protocol.

### Step 1 — Build a small pagination simulator

In [21]:
PAGES = {
    "start": (["A", "B"], "page-2"),
    "page-2": ([], "page-3"),
    "page-3": (["C"], None),
}

fetch_count = 0

def fetch_page(token: Optional[str]) -> Optional[Tuple[List[str], Optional[str]]]:
    global fetch_count
    fetch_count += 1

    if token is None:
        return None
    return PAGES[token]

### Step 2 — Use the returned page to control the loop

The initial token is `"start"`. Each successful fetch supplies the token for the
next iteration.

In [22]:
def collect_all_pages() -> Tuple[List[str], int]:
    global fetch_count
    fetch_count = 0

    token = "start"
    collected = []

    while (page := fetch_page(token)) is not None:
        items, token = page
        collected.extend(items)

    return collected, fetch_count


page_items, calls = collect_all_pages()
page_items, calls

(['A', 'B', 'C'], 4)

### Step 3 — Verify that the empty page did not stop iteration

In [23]:
assert page_items == ["A", "B", "C"]
assert calls == 4  # three pages plus the final fetch using token=None

print("Problem 6 checks passed.")

Problem 6 checks passed.


### Discussion

The important condition is:

```python
while (page := fetch_page(token)) is not None:
```

It says exactly what the API contract says: `None` marks exhaustion.

The expression also guarantees one fetch per iteration. Repeating the fetch in a
condition and body would be a serious bug because a page-fetch operation changes
the state of the traversal.

# Part 4 — Short-Circuiting and Capturing a Witness

## Problem 7 — Return the First Validation Failure

A validator returns:

- `None` for a valid record,
- a dictionary describing the violation for an invalid record.

We want to stop at the first invalid record and return its violation.

The built-in `any` function already stops at the first truthy value. The
challenge is retaining the value that caused it to stop.

### Step 1 — Define records and a validator

In [24]:
records = [
    {"id": 1, "email": "one@example.com", "age": 30},
    {"id": 2, "email": "two@example.com", "age": 17},
    {"id": 3, "email": "", "age": 40},
]

def validate_record(record: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    if record["age"] < 18:
        return {
            "id": record["id"],
            "field": "age",
            "reason": "must be at least 18",
        }

    if not record["email"]:
        return {
            "id": record["id"],
            "field": "email",
            "reason": "must not be empty",
        }

    return None

### Step 2 — Establish the clear loop-based solution

In [25]:
def first_violation_loop(items: Iterable[Dict[str, Any]]) -> Optional[Dict[str, Any]]:
    for item in items:
        violation = validate_record(item)
        if violation is not None:
            return violation
    return None


first_violation_loop(records)

{'id': 2, 'field': 'age', 'reason': 'must be at least 18'}

### Step 3 — Capture the witness used by `any`

The generator assigns each validator result to `violation`, then tests whether
that result is not `None`.

When a violation appears, `any` stops immediately. At that moment, `violation`
still refers to the object that caused the truthy test.

In [26]:
def first_violation_any(items: Iterable[Dict[str, Any]]) -> Optional[Dict[str, Any]]:
    if any(
        (violation := validate_record(item)) is not None
        for item in items
    ):
        return violation

    return None


first_violation_any(records)

{'id': 2, 'field': 'age', 'reason': 'must be at least 18'}

### Step 4 — Test short-circuiting and empty input

In [27]:
validation_calls = 0

def counted_validate(record: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    global validation_calls
    validation_calls += 1
    return validate_record(record)

def first_counted_violation(items: Iterable[Dict[str, Any]]) -> Optional[Dict[str, Any]]:
    if any(
        (violation := counted_validate(item)) is not None
        for item in items
    ):
        return violation
    return None


validation_calls = 0
found = first_counted_violation(records)

assert found == {
    "id": 2,
    "field": "age",
    "reason": "must be at least 18",
}
assert validation_calls == 2
assert first_counted_violation([]) is None

print("Problem 7 checks passed.")

Problem 7 checks passed.


### Readability Decision

This pattern is advanced and useful, but the ordinary `for` loop is easier for
many readers.

Use the `any` plus assignment-expression pattern when:

- short-circuit behavior is central,
- the captured witness is immediately returned or used,
- the validator's contract is simple,
- the team is comfortable with generator evaluation.

Otherwise, prefer the loop. Fewer lines do not automatically mean clearer code.

## Problem 8 — Understand `and` Evaluation with Dependent Assignments

Assignment expressions interact with short-circuiting. A later assignment may
never happen if an earlier condition is false.

We will trace the evaluation order rather than relying on intuition.

### Step 1 — Build a tracing helper

In [28]:
def traced_value(trace: List[str], label: str, value: Any) -> Any:
    trace.append(label)
    return value

### Step 2 — Run a condition whose first part succeeds

In [29]:
trace_one = []

condition_one = (
    (left := traced_value(trace_one, "left evaluated", 5)) > 0
    and
    (right := traced_value(trace_one, "right evaluated", left * 2)) > 20
)

condition_one, left, right, trace_one

(False, 5, 10, ['left evaluated', 'right evaluated'])

Both assignments occur because the first comparison is true. The overall
condition is false because `10 > 20` is false.

### Step 3 — Run a condition whose first part fails

We place the experiment inside a function so we can inspect whether the second
name exists in local scope.

In [30]:
def short_circuit_experiment() -> Tuple[bool, List[str], bool]:
    trace = []

    condition = (
        (first := traced_value(trace, "first evaluated", -1)) > 0
        and
        (second := traced_value(trace, "second evaluated", 999)) > 0
    )

    return condition, trace, "second" in locals()


short_circuit_experiment()

(False, ['first evaluated'], False)

### Step 4 — State the rule precisely

In [31]:
condition, trace, second_was_bound = short_circuit_experiment()

assert condition is False
assert trace == ["first evaluated"]
assert second_was_bound is False

print("Problem 8 checks passed.")

Problem 8 checks passed.


The second assignment expression is not evaluated when the first comparison is
false.

This matters when code later assumes that every walrus target exists. A name
bound only in a conditional branch should be used only where that branch
guarantees it was assigned.

# Part 5 — Precedence and Syntax

## Problem 9 — Detect a Precedence Bug

Consider these two conditions:

```python
if (count := len(items)) > 3:
    ...

if count := len(items) > 3:
    ...
```

They look similar, but they assign different values.

### Step 1 — Evaluate the parenthesized version

In [32]:
items = ["a", "b", "c", "d"]

if (count_correct := len(items)) > 3:
    correct_branch = True
else:
    correct_branch = False

count_correct, type(count_correct), correct_branch

(4, int, True)

### Step 2 — Evaluate the unparenthesized version

In an `if` statement, assignment expressions do not always require surrounding
parentheses. That does not mean parentheses are unnecessary for meaning.

The comparison binds more tightly than `:=`, so the comparison is evaluated
first.

In [33]:
if count_bug := len(items) > 3:
    bug_branch = True
else:
    bug_branch = False

count_bug, type(count_bug), bug_branch

(True, bool, True)

### Step 3 — Confirm the difference

In [34]:
assert count_correct == 4
assert type(count_correct) is int

assert count_bug is True
assert type(count_bug) is bool

print("Problem 9 checks passed.")

Problem 9 checks passed.


### Best Practice

When the assigned value participates in a larger comparison, parenthesize the
assignment expression:

```python
if (count := len(items)) > limit:
    ...
```

The parentheses communicate that `count` receives `len(items)`, not the Boolean
result of the comparison.

## Problem 10 — Explore Syntax Restrictions Safely

A standalone assignment expression is not accepted without parentheses:

```python
value := 10
```

We will use `compile` so the notebook can demonstrate the syntax error without
interrupting execution.

In [35]:
syntax_examples = {
    "standalone_without_parentheses": "value := 10",
    "standalone_with_parentheses": "(value := 10)",
    "inside_if": "if value := 10:\n    pass",
}

syntax_results = {}

for label, source in syntax_examples.items():
    try:
        compile(source, f"<{label}>", "exec")
    except SyntaxError as exc:
        syntax_results[label] = f"SyntaxError: {exc.msg}"
    else:
        syntax_results[label] = "valid"

syntax_results

{'standalone_without_parentheses': 'SyntaxError: invalid syntax',
 'standalone_with_parentheses': 'valid',
 'inside_if': 'valid'}

### Another restriction — do not rebind a comprehension loop variable

The assignment-expression target may not be the same name as a comprehension's
iteration variable.

In [36]:
illegal_comprehension = "[(number := number + 1) for number in range(3)]"

try:
    compile(illegal_comprehension, "<illegal-comprehension>", "exec")
except SyntaxError as exc:
    comprehension_error = exc.msg
else:
    comprehension_error = "unexpectedly valid"

comprehension_error

"assignment expression cannot rebind comprehension iteration variable 'number'"

In [37]:
assert syntax_results["standalone_without_parentheses"].startswith("SyntaxError")
assert syntax_results["standalone_with_parentheses"] == "valid"
assert syntax_results["inside_if"] == "valid"
assert comprehension_error != "unexpectedly valid"

print("Problem 10 checks passed.")

Problem 10 checks passed.


### Discussion

The syntax rules are designed to reduce ambiguity and prevent confusing
interactions with comprehension targets.

For production code, avoid trying to memorize every exceptional grammar rule.
Use assignment expressions in conventional positions:

- an `if` condition,
- a `while` condition,
- a comprehension filter,
- a parenthesized subexpression.

When unsure, favor an ordinary assignment.

# Part 6 — Scope and Lifetime

## Problem 11 — Observe Comprehension Scope

Comprehension loop variables do not leak into the surrounding scope in Python 3.

Assignment-expression targets inside comprehensions behave differently: they bind
in the containing scope.

This difference can surprise readers.

### Step 1 — Run the comprehension inside a function

In [38]:
def comprehension_scope_demo() -> Dict[str, Any]:
    words = ["cat", "python", "api", "notebook"]

    selected_lengths = [
        length
        for word in words
        if (length := len(word)) >= 5
    ]

    return {
        "selected_lengths": selected_lengths,
        "final_length": length,
        "word_is_local": "word" in locals(),
        "length_is_local": "length" in locals(),
    }


scope_result = comprehension_scope_demo()
scope_result

{'selected_lengths': [6, 8],
 'final_length': 8,
 'word_is_local': False,
 'length_is_local': True}

The final value of `length` corresponds to the final input word, even though that
word passes the filter here. More generally, the target is updated for each
evaluation of the filter, including evaluations that fail.

### Step 2 — Use data where the final item fails the filter

In [39]:
def final_failed_item_demo() -> Tuple[List[int], int]:
    words = ["python", "notebook", "api"]

    selected = [
        size
        for word in words
        if (size := len(word)) >= 5
    ]

    return selected, size


final_failed_item_demo()

([6, 8], 3)

In [40]:
assert scope_result == {
    "selected_lengths": [6, 8],
    "final_length": 8,
    "word_is_local": False,
    "length_is_local": True,
}

selected, final_size = final_failed_item_demo()
assert selected == [6, 8]
assert final_size == 3

print("Problem 11 checks passed.")

Problem 11 checks passed.


### Best Practice

Do not rely on a walrus target's post-comprehension value unless that behavior is
the explicit point of the code.

The target's main purpose should be local reuse within the comprehension. If the
value matters afterward, an ordinary loop usually communicates lifetime and
intent more clearly.

# Part 7 — Structured Parsing

## Problem 12 — Parse an INI-Like Document

We will parse a small document with:

- section headers such as `[server]`,
- key-value pairs such as `host=localhost`,
- blank lines and comments,
- malformed lines that must be reported.

A regular-expression match object should be evaluated only once per attempted
pattern.

### Step 1 — Define the document and patterns

In [41]:
document_lines = [
    "# application configuration",
    "[server]",
    "host = localhost",
    "port = 8080",
    "",
    "[database]",
    "name = analytics",
    "pool_size = 5",
    "this line is malformed",
]

section_pattern = re.compile(r"\[(?P<section>[A-Za-z_][A-Za-z0-9_]*)\]")
pair_pattern = re.compile(
    r"(?P<key>[A-Za-z_][A-Za-z0-9_]*)\s*=\s*(?P<value>.*)"
)

### Step 2 — Plan the state machine

The parser has one piece of state: `current_section`.

For each stripped line:

1. ignore comments and blanks,
2. try a section match,
3. otherwise try a key-value match,
4. reject a pair that appears before any section,
5. report anything else as malformed.

### Step 3 — Implement the parser

The assignment expressions keep the match operations next to the branches they
control.

In [42]:
def parse_document(lines: Iterable[str]) -> Tuple[Dict[str, Dict[str, str]], List[str]]:
    config: Dict[str, Dict[str, str]] = {}
    errors: List[str] = []
    current_section: Optional[str] = None

    for line_number, raw_line in enumerate(lines, start=1):
        line = raw_line.strip()

        if not line or line.startswith("#"):
            continue

        if section_match := section_pattern.fullmatch(line):
            current_section = section_match.group("section")
            config.setdefault(current_section, {})
            continue

        if pair_match := pair_pattern.fullmatch(line):
            if current_section is None:
                errors.append(
                    f"line {line_number}: key-value pair before any section"
                )
                continue

            key = pair_match.group("key")
            value = pair_match.group("value")
            config[current_section][key] = value
            continue

        errors.append(f"line {line_number}: malformed content")

    return config, errors


parsed_config, parse_errors = parse_document(document_lines)
parsed_config, parse_errors

({'server': {'host': 'localhost', 'port': '8080'},
  'database': {'name': 'analytics', 'pool_size': '5'}},
 ['line 9: malformed content'])

### Step 4 — Verify the complete structure

In [43]:
assert parsed_config == {
    "server": {
        "host": "localhost",
        "port": "8080",
    },
    "database": {
        "name": "analytics",
        "pool_size": "5",
    },
}

assert parse_errors == [
    "line 9: malformed content",
]

print("Problem 12 checks passed.")

Problem 12 checks passed.


### Discussion

The conditions

```python
if section_match := section_pattern.fullmatch(line):
```

and

```python
if pair_match := pair_pattern.fullmatch(line):
```

are conventional because a match object is truthy and `None` means no match.

The names are specific. Calling both targets merely `match` would be shorter but
less descriptive in a stateful parser.

# Part 8 — Safe Parsing Helpers and Comprehensions

## Problem 13 — Parse JSON Events and Filter Them

`json.loads` raises an exception for malformed input. Assignment expressions
cannot catch exceptions, so first we need a helper that converts a parsing
failure into a normal value.

Then we can use `:=` to parse each line exactly once in a comprehension.

### Step 1 — Define event lines

In [44]:
event_lines = [
    '{"type": "purchase", "amount": 40, "user": "u1"}',
    'not json',
    '{"type": "view", "user": "u2"}',
    '{"type": "purchase", "amount": 0, "user": "u3"}',
    '{"type": "purchase", "amount": 75, "user": "u4"}',
    '[]',
]

### Step 2 — Build a safe parser

The helper accepts only JSON objects. Other JSON values, such as arrays, are not
valid events for this problem.

In [45]:
json_parse_calls = 0

def parse_event(line: str) -> Optional[Dict[str, Any]]:
    global json_parse_calls
    json_parse_calls += 1

    try:
        value = json.loads(line)
    except json.JSONDecodeError:
        return None

    if not isinstance(value, dict):
        return None

    return value

### Step 3 — Parse once and filter purchases

A purchase with amount `0` is still a valid purchase, so we must not use the
amount's truthiness to decide whether the event exists.

In [46]:
json_parse_calls = 0

purchases = [
    event
    for line in event_lines
    if (event := parse_event(line)) is not None
    and event.get("type") == "purchase"
    and isinstance(event.get("amount"), (int, float))
]

purchases, json_parse_calls

([{'type': 'purchase', 'amount': 40, 'user': 'u1'},
  {'type': 'purchase', 'amount': 0, 'user': 'u3'},
  {'type': 'purchase', 'amount': 75, 'user': 'u4'}],
 6)

### Step 4 — Aggregate the accepted events

In [47]:
total_purchase_amount = sum(event["amount"] for event in purchases)

assert purchases == [
    {"type": "purchase", "amount": 40, "user": "u1"},
    {"type": "purchase", "amount": 0, "user": "u3"},
    {"type": "purchase", "amount": 75, "user": "u4"},
]
assert total_purchase_amount == 115
assert json_parse_calls == len(event_lines)

print("Problem 13 checks passed.")

Problem 13 checks passed.


### Discussion

The helper separates exception handling from expression-level filtering.

This separation is a useful design pattern:

1. convert exceptional parsing into `None` or a result object,
2. use an assignment expression to avoid repeating the safe parse,
3. use explicit checks for values whose truthiness is meaningful data.

# Part 9 — Iterative Algorithms

## Problem 14 — Generate Batches Until the Source Is Exhausted

We have an iterator of records and want to yield fixed-size batches. The final
batch may be smaller.

A helper returns either:

- a non-empty list representing the next batch,
- `None` when no records remain.

The caller needs the returned batch both for loop control and for processing.

### Step 1 — Write the batch reader

In [48]:
def read_batch(source: Iterator[Any], batch_size: int) -> Optional[List[Any]]:
    batch = []

    for _ in range(batch_size):
        item = next(source, MISSING)
        if item is MISSING:
            break
        batch.append(item)

    return batch if batch else None

### Step 2 — Build the generator with an assignment expression

In [49]:
def batched(source: Iterable[Any], batch_size: int) -> Iterator[List[Any]]:
    if batch_size <= 0:
        raise ValueError("batch_size must be positive")

    iterator = iter(source)

    while (batch := read_batch(iterator, batch_size)) is not None:
        yield batch

### Step 3 — Test several boundary cases

In [50]:
assert list(batched([], 3)) == []
assert list(batched([1], 3)) == [[1]]
assert list(batched([1, 2, 3], 3)) == [[1, 2, 3]]
assert list(batched([1, 2, 3, 4, 5, 6, 7], 3)) == [
    [1, 2, 3],
    [4, 5, 6],
    [7],
]

try:
    list(batched([1, 2], 0))
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for a non-positive batch size")

print("Problem 14 checks passed.")

Problem 14 checks passed.


### Discussion

The protocol deliberately returns `None` for exhaustion rather than `[]`.
That makes the interface unambiguous even if the domain could someday allow an
empty batch as meaningful data.

The assignment expression keeps the acquisition operation and exhaustion check
together:

```python
while (batch := read_batch(...)) is not None:
```

# Part 10 — Refactoring Dense Walrus Code

## Problem 15 — Decide When Not to Use `:=`

The following style is technically possible, but it asks the reader to track too
many temporary names and conditions at once:

```python
result = [
    (name, score)
    for raw in rows
    if (clean := raw.strip())
    and (parts := clean.split(":"))
    and len(parts) == 2
    and (name := parts[0].strip())
    and (score_text := parts[1].strip()).isdigit()
    and (score := int(score_text)) >= 70
]
```

The problem is to rewrite this logic so that validation steps are explicit and
testable.

### Step 1 — Define the data

In [51]:
score_rows = [
    "Ada: 95",
    "Grace: 88",
    "invalid row",
    "Linus: sixty",
    ": 100",
    "Guido: 69",
    "Margaret: 100",
]

### Step 2 — Move parsing into a focused helper

The helper returns either a validated `(name, score)` tuple or `None`.

Inside the helper, ordinary assignments are clearer because the algorithm has
multiple sequential validation steps.

In [52]:
def parse_passing_score(raw: str) -> Optional[Tuple[str, int]]:
    clean = raw.strip()
    if not clean:
        return None

    parts = clean.split(":")
    if len(parts) != 2:
        return None

    name = parts[0].strip()
    score_text = parts[1].strip()

    if not name or not score_text.isdigit():
        return None

    score = int(score_text)
    if score < 70:
        return None

    return name, score

### Step 3 — Use one assignment expression at the boundary

Now the comprehension has one simple responsibility: retain successful parse
results.

In [53]:
passing_scores = [
    parsed
    for row in score_rows
    if (parsed := parse_passing_score(row)) is not None
]

passing_scores

[('Ada', 95), ('Grace', 88), ('Margaret', 100)]

In [54]:
assert passing_scores == [
    ("Ada", 95),
    ("Grace", 88),
    ("Margaret", 100),
]

print("Problem 15 checks passed.")

Problem 15 checks passed.


### Lesson

A good assignment expression often replaces one repeated operation.

A bad assignment-expression chain may compress an entire algorithm into a Boolean
maze.

Use helper functions to give multi-step transformations names, contracts, and
independent tests.

# Part 11 — Advanced Integrated Problems

## Problem 16 — Find the First Window Whose Average Exceeds a Limit

A sensor produces numeric readings. We inspect consecutive windows of three
readings.

For each window, computing the average is required both for the threshold test and
for the returned diagnostic record.

We want the first window whose average is greater than `70`.

### Step 1 — Build overlapping windows

In [55]:
def windows(values: List[float], size: int) -> Iterator[List[float]]:
    if size <= 0:
        raise ValueError("size must be positive")

    for start in range(len(values) - size + 1):
        yield values[start:start + size]

### Step 2 — Capture the average inside the condition

In [56]:
def first_hot_window(
    readings: List[float],
    window_size: int,
    limit: float,
) -> Optional[Dict[str, Any]]:
    for index, window in enumerate(windows(readings, window_size)):
        if (average := sum(window) / len(window)) > limit:
            return {
                "start": index,
                "window": window,
                "average": average,
            }

    return None

### Step 3 — Test exact-threshold and above-threshold behavior

In [57]:
sensor_readings = [50, 60, 70, 80, 90]

hot_window = first_hot_window(sensor_readings, 3, 70)
exact_limit = first_hot_window([60, 70, 80], 3, 70)

assert hot_window == {
    "start": 2,
    "window": [70, 80, 90],
    "average": 80.0,
}
assert exact_limit is None

hot_window

{'start': 2, 'window': [70, 80, 90], 'average': 80.0}

### Discussion

The ordinary alternative is also clear:

```python
average = sum(window) / len(window)
if average > limit:
    ...
```

The walrus version is reasonable because the calculation is short, the name is
descriptive, and the value is used immediately in the successful branch.

This is a judgment call rather than a mandatory refactoring.

## Problem 17 — Build a Retry Loop Without Repeating the Attempt

A simulated operation produces a sequence of outcomes:

- `("retry", reason)` means try again,
- `("success", payload)` means return the payload,
- `("fatal", reason)` means stop with an error,
- `None` means the attempt source itself is exhausted.

Calling the operation twice in one iteration would consume two outcomes and
corrupt the control flow.

### Step 1 — Create the simulator

In [58]:
def make_attempt(outcomes: Iterable[Optional[Tuple[str, Any]]]):
    iterator = iter(outcomes)
    calls = {"count": 0}

    def attempt() -> Optional[Tuple[str, Any]]:
        calls["count"] += 1
        return next(iterator, None)

    return attempt, calls

### Step 2 — Implement the retry protocol

In [59]:
def run_with_retries(attempt, max_retries: int) -> Any:
    retries = 0

    while (outcome := attempt()) is not None:
        status, value = outcome

        if status == "success":
            return value

        if status == "fatal":
            raise RuntimeError(value)

        if status != "retry":
            raise ValueError(f"unknown status: {status}")

        retries += 1
        if retries > max_retries:
            raise TimeoutError("retry limit exceeded")

    raise EOFError("attempt source exhausted")

### Step 3 — Verify call count and payload preservation

The successful payload is an empty dictionary. It is falsy, but it is still a
valid result. The loop tests the entire outcome against `None`, not the payload's
truthiness.

In [60]:
attempt, call_counter = make_attempt([
    ("retry", "temporary congestion"),
    ("retry", "rate limited"),
    ("success", {}),
])

payload = run_with_retries(attempt, max_retries=3)

assert payload == {}
assert call_counter["count"] == 3

print("Problem 17 checks passed.")

Problem 17 checks passed.


### Discussion

This example highlights two principles:

1. Store state-changing results exactly once.
2. Compare with the protocol sentinel explicitly.

The assignment expression is valuable because `attempt()` advances the simulated
operation. Duplicate evaluation would not merely be inefficient; it would change
the program's meaning.

## Problem 18 — Build a Small Event-Processing Pipeline

We will combine several ideas:

- safe JSON parsing,
- assignment in a comprehension filter,
- explicit handling of `0`,
- grouped aggregation,
- a first-failure diagnostic.

An event is accepted when:

- it is a JSON object,
- its `"kind"` is `"metric"`,
- `"name"` is a non-empty string,
- `"value"` is an integer or float but not a Boolean.

### Step 1 — Define the raw event stream

In [61]:
raw_metric_events = [
    '{"kind": "metric", "name": "requests", "value": 10}',
    '{"kind": "metric", "name": "errors", "value": 0}',
    '{"kind": "message", "name": "note", "value": 7}',
    '{"kind": "metric", "name": "", "value": 4}',
    '{"kind": "metric", "name": "requests", "value": 15}',
    '{"kind": "metric", "name": "healthy", "value": true}',
    'broken json',
]

### Step 2 — Write a validator that returns a reason

Returning a reason rather than only `True` or `False` makes diagnostics possible.

In [62]:
def metric_violation(event: Dict[str, Any]) -> Optional[str]:
    if event.get("kind") != "metric":
        return "not a metric event"

    name = event.get("name")
    if not isinstance(name, str) or not name:
        return "metric name must be a non-empty string"

    value = event.get("value")
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        return "metric value must be numeric and not Boolean"

    return None

### Step 3 — Parse once and retain only valid metrics

We deliberately call `metric_violation` only after parsing has succeeded.

In [63]:
valid_metrics = [
    event
    for raw in raw_metric_events
    if (event := parse_event(raw)) is not None
    and metric_violation(event) is None
]

valid_metrics

[{'kind': 'metric', 'name': 'requests', 'value': 10},
 {'kind': 'metric', 'name': 'errors', 'value': 0},
 {'kind': 'metric', 'name': 'requests', 'value': 15}]

### Step 4 — Aggregate totals by metric name

In [64]:
metric_totals = Counter()

for event in valid_metrics:
    metric_totals[event["name"]] += event["value"]

dict(metric_totals)

{'requests': 25, 'errors': 0}

### Step 5 — Verify zero preservation and invalid-value rejection

In [65]:
assert valid_metrics == [
    {"kind": "metric", "name": "requests", "value": 10},
    {"kind": "metric", "name": "errors", "value": 0},
    {"kind": "metric", "name": "requests", "value": 15},
]

assert dict(metric_totals) == {
    "requests": 25,
    "errors": 0,
}

print("Problem 18 checks passed.")

Problem 18 checks passed.


# Part 12 — Additional Practice Problems with Solutions

## Practice Problem A — Capture a Search Result

Write a function that scans strings and returns the first integer embedded in any
string. Use a compiled regular expression and avoid calling `.search()` twice.

For this problem, `None` means that no integer exists.

### Solution

In [66]:
integer_pattern = re.compile(r"-?\d+")

def first_embedded_integer(values: Iterable[str]) -> Optional[int]:
    for value in values:
        if match := integer_pattern.search(value):
            return int(match.group())
    return None


assert first_embedded_integer(["none", "x=-12", "99"]) == -12
assert first_embedded_integer(["none", "still none"]) is None

print("Practice A checks passed.")

Practice A checks passed.


## Practice Problem B — Filter Derived Ratios

Given `(numerator, denominator)` pairs, compute ratios only when the denominator
is nonzero and the ratio is at least `1.5`.

Calculate each accepted ratio exactly once.

### Solution

In [67]:
pairs = [
    (3, 2),
    (1, 0),
    (10, 4),
    (2, 4),
    (9, 6),
]

ratios = [
    ratio
    for numerator, denominator in pairs
    if denominator != 0
    and (ratio := numerator / denominator) >= 1.5
]

assert ratios == [1.5, 2.5, 1.5]

ratios

[1.5, 2.5, 1.5]

The denominator check appears first. Because `and` short-circuits, division is
never attempted when the denominator is zero.

## Practice Problem C — Stop at the First Empty Data Block

A generator yields lists. Print or collect blocks until the first empty list.

Here an empty list is deliberately the sentinel, so a truthiness-based loop is
appropriate.

### Solution

In [68]:
def block_source() -> Iterator[List[int]]:
    yield [1, 2]
    yield [3]
    yield []
    yield [4, 5]  # must not be consumed by the loop body


source = block_source()
collected_blocks = []

while block := next(source):
    collected_blocks.append(block)

assert collected_blocks == [[1, 2], [3]]

collected_blocks

[[1, 2], [3]]

This example differs from the earlier configuration and pagination problems.
Here the domain explicitly defines an empty list as the stopping value. Therefore,
truthiness expresses the protocol correctly.

## Practice Problem D — Preserve the Final Failed Transformation

Transform integers by subtracting ten. Keep transformed values that are positive.
Then inspect the final assigned transformation.

This exercise reinforces that the walrus target is updated even when the filter
fails.

### Solution

In [69]:
source_numbers = [15, 12, 8]

positive_transforms = [
    transformed
    for number in source_numbers
    if (transformed := number - 10) > 0
]

assert positive_transforms == [5, 2]
assert transformed == -2

positive_transforms, transformed

([5, 2], -2)

Relying on the final `transformed` value would normally be poor style. The value
is shown only to make comprehension evaluation behavior visible.

## Practice Problem E — Replace an Overly Clever Expression

Review this expression:

```python
while (line := next(lines, None)) is not None and (clean := line.strip()) != "END":
    ...
```

It combines exhaustion, normalization, and sentinel detection in one condition.
Rewrite it if the loop body also needs to handle blank lines differently.

### Solution

Once blank-line behavior becomes a separate branch, a less compressed loop is
clearer.

In [70]:
def collect_until_end(lines: Iterator[str]) -> List[str]:
    collected = []

    while (line := next(lines, None)) is not None:
        clean = line.strip()

        if not clean:
            continue

        if clean == "END":
            break

        collected.append(clean)

    return collected


assert collect_until_end(iter([
    " alpha ",
    "   ",
    "beta",
    "END",
    "ignored",
])) == ["alpha", "beta"]

print("Practice E checks passed.")

Practice E checks passed.


# Final Review

## A Practical Decision Checklist

Before introducing an assignment expression, ask:

1. **Is the expression otherwise evaluated more than once?**
   Repeated parsing, matching, reading, fetching, or transformation is a strong
   reason to consider `:=`.

2. **Does the value naturally control the surrounding condition?**
   Reading until a sentinel and matching a pattern are conventional examples.

3. **Is the assigned name used immediately?**
   A nearby use is easier to understand than a name whose significance appears
   many lines later.

4. **Could a falsy value still be valid data?**
   Use explicit comparisons such as `is not None` or a unique sentinel.

5. **Are parentheses needed to communicate precedence?**
   Prefer `(count := len(items)) > limit`.

6. **Does short-circuiting guarantee that the name was assigned?**
   Do not use a later target when an earlier `and` condition may have skipped it.

7. **Would a helper function or ordinary loop be clearer?**
   Assignment expressions should remove accidental repetition, not compress an
   entire algorithm into one expression.

## Common Good Patterns

### Match once

```python
if match := pattern.search(text):
    use(match)
```

### Read until a truthy sentinel

```python
while chunk := stream.read(size):
    process(chunk)
```

### Read until an explicit sentinel

```python
while (item := get_item()) is not None:
    process(item)
```

### Compute once in a comprehension filter

```python
[
    transformed
    for item in items
    if (transformed := transform(item)) is not None
]
```

### Capture a short-circuit witness

```python
if any((error := validate(item)) is not None for item in items):
    return error
```

## Patterns That Deserve Extra Caution

### Accidental Boolean assignment

```python
if count := len(items) > limit:
    ...
```

Prefer:

```python
if (count := len(items)) > limit:
    ...
```

### Treating valid falsy data as missing

```python
if timeout := config.get("timeout"):
    ...
```

This rejects `0`.

### Depending on post-comprehension target state

```python
values = [x for item in items if (x := transform(item))]
use(x)
```

The final `x` may come from an item that failed the filter.

### Long chains of assignments

If a reader must track several temporary names across many `and` clauses, move
the logic into a helper function or use ordinary statements.

## Final Notebook Verification

The following cell summarizes the major outputs created in the notebook.

In [71]:
summary = {
    "commands": walrus_commands,
    "normalized_labels": walrus_version,
    "important_log_records": len(important_records),
    "paginated_items": page_items,
    "first_violation_id": found["id"],
    "parsed_sections": sorted(parsed_config),
    "purchase_total": total_purchase_amount,
    "passing_scores": passing_scores,
    "metric_totals": dict(metric_totals),
}

summary

{'commands': ['STATUS', 'deploy api', 'ReStart Worker'],
 'normalized_labels': ['alpha team', 'data platform', 'customer success'],
 'important_log_records': 3,
 'paginated_items': ['A', 'B', 'C'],
 'first_violation_id': 2,
 'parsed_sections': ['database', 'server'],
 'purchase_total': 115,
 'passing_scores': [('Ada', 95), ('Grace', 88), ('Margaret', 100)],
 'metric_totals': {'requests': 25, 'errors': 0}}

## Closing Thought

The walrus operator is most effective when it makes evaluation order visible:

> compute or acquire one value, test that same value, and reuse it nearby.

When it obscures the sequence of operations, ordinary assignments are usually
better. Advanced Python is not about maximizing syntax density; it is about
choosing syntax that preserves correctness and makes intent easy to inspect.